Code to run a RNN model with 5-fold cross validation to predict the length of ITI from neural activity data. Neural activity data is fed in as a excel with different Time points labeled, and activity from both the preceding and subsequent trial to the ITI are used (next trial data is labeled Time_X_next_trial in the excel or CSV file).





In [ ]:
# imports and setting a seed

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GRU
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import AdamW


SEED = 42425  # setting a seed to ensure randomness

Architecture of the GRU model used to make predictions; this can be changed by changing the layers. Dropout used after each layer, and dropout and learning date were optimized using a grid search.

In [ ]:
def build_gru_model(input_dim, output_dim, dropout_rate=0.3, learning_rate=0.001):
    model = Sequential()
    model.add(GRU(256, return_sequences=True, input_shape=(2, input_dim)))
    model.add(Dropout(dropout_rate))
    model.add(GRU(128, return_sequences=True))
    model.add(Dropout(dropout_rate))
    model.add(GRU(64))
    model.add(Dropout(dropout_rate))
    model.add(Dense(32, activation="relu"))
    model.add(Dropout(dropout_rate))
    model.add(Dense(output_dim, activation="softmax"))

    optimizer = AdamW(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

Code to run the loop with 5-fold cross validation.

Code runs on all clusters first, and then individually drops out clusters, and creates metrics files in separate folders for each run. DropOut models are equalized by number of cells (through random sampling of the number of cells in the smallest cluster) before they are fed in, since some clusters in the initial dataset had significantly more cells, and thus, this approach avoids the amount of data influencing the cluster by cluster results

In [ ]:
def crossval_gru_full_model_cluster_dropout(df_long):

    ohe = OneHotEncoder(sparse_output=False) #one hot encoder to convert categorical data into binary form

    df_long["Cluster"] = np.random.randint(0, 5, size=len(df_long))

    df_long = df_long.copy()  #df long is a input, it is a excel/csv with various time points and activity, and will also typically include a column called 'Cluster'
    rng = np.random.RandomState(SEED)

    df_long["ITI_bin"] = df_long.groupby("Animal_ID")["ITI"].transform(
        lambda x: pd.qcut(x, q=5, labels=False, duplicates="drop") # converting ITI from a continuous variable (inter trial interval time) to a categorical variable (i.e. top 20% fastest, next 20% etc.)
    )

    # since we use both the previous trial and the subsequent trial to compute ITI, this adds columns (features) for what activity will be in the next trial

    time_cols = [
        col for col in df_long.columns
        if col.startswith("Time_") and not col.endswith("next_trial")
    ]

    for col in time_cols:
        df_long[f"{col}_next_trial"] = df_long.groupby("Cell_ID")[col].shift(-1)

    df_cleaned = df_long.dropna(
        subset=[f"{col}_next_trial" for col in time_cols] #clean things up
    )


    # equalize cells to that of the lowest cluster by randomly subsampling from all other clusters
    # This step should be removed if the run is not a cluster by cluster dropout and is instead being run for all cells across all clusters

    cluster_cell_counts = df_cleaned.groupby("Cluster")["Cell_ID"].nunique()
    min_cells = cluster_cell_counts.min()

    sampled_cells = (
        df_cleaned.groupby("Cluster")["Cell_ID"]
        .unique()
        .apply(
            lambda cell_ids: np.random.RandomState(SEED).choice(
                cell_ids, size=min_cells, replace=False
            )
        )
    )

    sampled_cells_flat = np.concatenate(sampled_cells.values)
    df_equalized_base = df_cleaned[
        df_cleaned["Cell_ID"].isin(sampled_cells_flat)
    ].copy()

    clusters = sorted(df_equalized_base["Cluster"].unique())


    # a loop to drop all the cells from each cluster one at a time. A new set of metrics will be saved for each cluster dropout
    for dropped_cluster in [None] + clusters:

        tag = "none" if dropped_cluster is None else str(dropped_cluster)
        output_dir = f"256-gru-DO_{tag}"
        os.makedirs(output_dir, exist_ok=True)

        print("\n")
        print(f"Cluster Dropout: {tag}")
        print("\n")

        df_equalized = df_equalized_base.copy()

        if dropped_cluster is not None:
            df_equalized = df_equalized[
                df_equalized["Cluster"] != dropped_cluster
            ].reset_index(drop=True)


        Y = ohe.fit_transform(df_equalized[["ITI_bin"]]) #one hot encoding the categorical ITI variable
        Y_labels = np.argmax(Y, axis=1)

        # reshaping the data for a GRU model. The model operates in time steps, and since the initial data used 2 distinct trials, the data is reshaped so that it can be fed in in 2 time steps (i.e. Trial 1 = time step 1 and Trial 2 = time step 2)
        X_now = df_equalized[time_cols].values.reshape(-1, 1, len(time_cols))
        X_next = df_equalized[
            [f"{col}_next_trial" for col in time_cols]
        ].values.reshape(-1, 1, len(time_cols))

        X_full = np.concatenate([X_now, X_next], axis=1)

        # Scale per timestep
        for t in range(X_full.shape[1]):
            scaler = StandardScaler()
            X_full[:, t, :] = scaler.fit_transform(X_full[:, t, :])

        #5-fold CV
        skf = StratifiedKFold(
            n_splits=5, shuffle=True, random_state=SEED
        )

        all_preds = []
        all_metrics = []

        for fold, (train_idx, test_idx) in enumerate(
            skf.split(X_full, Y_labels), 1
        ):
            print(f"\n=== Fold {fold}/5 ===")

            X_train, X_test = X_full[train_idx], X_full[test_idx]
            Y_train, Y_test = Y[train_idx], Y[test_idx]

            model = build_gru_model(
                input_dim=X_full.shape[2],
                output_dim=Y.shape[1]
            )

            early_stop = EarlyStopping(
                monitor="val_loss",
                patience=10,
                restore_best_weights=True
            )

            model.fit(
                X_train,
                Y_train,
                validation_split=0.1, # 10% for validation set (separate from the 20% held out dataset)
                epochs=10000,
                batch_size=32,
                verbose=0,
                callbacks=[early_stop] #early stopping to stop training when val loss is stable
            )

            Y_pred = model.predict(X_test)
            pred_labels = np.argmax(Y_pred, axis=1)
            true_labels = np.argmax(Y_test, axis=1)

            #obtaining, calculating, storing results

            acc = accuracy_score(true_labels, pred_labels)

            if 0 in true_labels:
                bin0_true = (true_labels == 0).astype(int)
                bin0_scores = Y_pred[:, 0]
                bin0_auroc = roc_auc_score(bin0_true, bin0_scores)
            else:
                bin0_auroc = np.nan

            class_report = classification_report(
                true_labels, pred_labels, output_dict=True
            )

            bin0_report = class_report.get(
                "0",
                {"precision": np.nan, "recall": np.nan, "f1-score": np.nan}
            )

            #storing data and metrics for each dropout in files that can be downloaded - each cluster dropout has a specific folder

            df_fold = df_equalized.iloc[test_idx].copy()
            df_fold["Predicted_Bin"] = pred_labels
            df_fold["Actual_Bin"] = true_labels
            df_fold["Correct"] = (pred_labels == true_labels).astype(int)
            df_fold["Fold"] = fold

            all_preds.append(
                df_fold[
                    ["Cell_ID", "Cluster",
                     "Predicted_Bin", "Actual_Bin",
                     "Correct", "Fold"]
                ]
            )

            all_metrics.append({
                "Fold": fold,
                "Accuracy": acc,
                "AUROC_Bin0": bin0_auroc,
                "Bin0_Precision": bin0_report["precision"],
                "Bin0_Recall": bin0_report["recall"],
                "Bin0_F1": bin0_report["f1-score"]
            })

            #print certain metrics

            print(f"Accuracy: {acc:.4f}")
            print(f"AUROC (Bin 0): {bin0_auroc:.4f}")

        df_all_preds = pd.concat(all_preds).reset_index(drop=True)
        df_all_metrics = pd.DataFrame(all_metrics)

        #convert the dataframe to excel so it could be downloaded
        df_all_preds.to_excel(
            os.path.join(output_dir, "AllCluster_Predictions.xlsx"),
            index=False
        )

        df_all_metrics.to_excel(
            os.path.join(output_dir, "AllCluster_Metrics.xlsx"),
            index=False
        )

        #at the end of each 5-fold CV, avg metrics that are stored in the df are printed so the script can be monitored.

        print("\n=== 5-Fold Cross-Validation Results ===")
        print(df_all_metrics.mean(numeric_only=True))


Running the script/loading in the data

In [ ]:
#loading in a excel file with the neural activity per time point and cluster to be used when running the script

df_long = pd.read_csv("/content/df_long_d8.csv") # replace this with the path to your excel/csv
print(df_long.shape)

#making sure all necessary columns exist
print(df_long.columns)


(220996, 101)
Index(['Animal_ID', 'Cell_ID', 'Trial', 'ITI', 'Time_0', 'Time_1', 'Time_2',
       'Time_3', 'Time_4', 'Time_5',
       ...
       'Time_87', 'Time_88', 'Time_89', 'Time_90', 'Time_91', 'Time_92',
       'Time_93', 'Time_94', 'Time_95', 'Time_96'],
      dtype='object', length=101)


In [ ]:
#call the functions and run the script

crossval_gru_full_model_cluster_dropout(df_long)
